In [1]:
import re

from sisyphus.utils.helper_functions import get_plain_articledb
from sisyphus.chain.label import BaseLabeler, Labeling, save_labeled_paras_wrapper
from sisyphus.chain.chain_elements import Filter, run_chains_with_extarction_history_multi_threads


class NLOCoefficientLabler(BaseLabeler):
    property = "nlo_coefficient"
    regex_pattern = re.compile(r'\b(dij|d(?:1[1-6]|2[1-6]|3[1-6]))\b')

class CutoffEdgeLabler(BaseLabeler):
    property = "cutoff_edge"
    regex_pattern = re.compile(r'\b(cutoff edge|absorption edge|transparency window|cutoff edges|λcutoff|cutoff edges|high transmittance|transmission spectral range)\b', re.I)
    
class Phase_matching_wavelengthLabler(BaseLabeler):
    property = "phase_matching_wavelength"
    regex_pattern = re.compile(r'\b(phase-matching)\b', re.I)

class LIDTLabler(BaseLabeler):
    property = "Laser-induced Damage Threshold"
    regex_pattern = re.compile(r'\b(laser-induced damage threshold|LIDT)\b', re.I)


labeler = Labeling()
labeler.add_labeler(NLOCoefficientLabler())
labeler.add_labeler(CutoffEdgeLabler())
labeler.add_labeler(Phase_matching_wavelengthLabler())
labeler.add_labeler(LIDTLabler())

database = get_plain_articledb('DUV_NLO_0015')
loader = Filter(database)


save_labeled_paras = save_labeled_paras_wrapper('DUV_NLO_0015_labeled')

chain = loader + labeler +save_labeled_paras

run_chains_with_extarction_history_multi_threads(
    chain=chain,
    directory='E:\\sisyphus_.0.2.0\\articles_processed',
    batch_size=10,
    namespace='DUV_NLO_0015_labeled'
)

100%|██████████| 15/15 [00:00<00:00, 63.43it/s]


In [1]:
from dotenv import load_dotenv
_ = load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.prompts.chat import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Dict

from sisyphus.chain.paragraph import Paragraph
from sisyphus.chain.extract import BaseExtractor, Extraction

from sisyphus.chain.paragraph import Paragraph
from sisyphus.chain.extract import BaseExtractor, Extraction

# output models
class NLOCoefficient(BaseModel):
    """Extract nonlinear optical coefficients from scientific papers."""
    dij: Optional[Dict[Optional[Literal['d11', 'd12', 'd13', 'd14', 'd15', 'd16','d21', 'd22','d23', 'd24', 'd25', 'd26','d31', 'd32','d33', 'd34', 'd35', 'd36']], Dict[Optional[float], Optional[str]]]] = Field(description="The specific second-order nonlinear polarizability tensor components with unit appearing in the text, e.g.,{'d33': {-1.19: 'pm/V'}, 'd31': {0.09: '× KDP'}}")
    deff: Optional[Dict[Optional[float], Optional[str]]] = Field(description="Effective nonlinear coefficient with unit, e.g., {3.2: 'pm/V'}")
    test_wavelength: Optional[str] = Field(description="The test wavelength of effective nonlinear coefficient(must be deff!) with unit, the arrangement order of test wavelengths corresponds to deff e.g., '1950nm, 1064nm'")

class MetaData(BaseModel):
    composition: str = Field(description='chemical formula of nonlinear optical crystals, e.g., "NH4B4O6F"')

class NLOCoefficientRecord(BaseModel):
    metadata: MetaData
    nlo_coefficient: NLOCoefficient

class Records(BaseModel):
    """You are supposed to extract all nonlinear optical coefficient materials, despite of their prominence in the text."""
    records: List[NLOCoefficientRecord]

# prompt
simple_prompt_template_no_syn = ChatPromptTemplate.from_messages([
    ('user',"""
You are required to extract material information from text provided below and output desired format which generally a list of dictionaries, and for each includes metadata and property information. Return empty list if no property found. Specifically, the metadata has structure as follows:
metadata: {{
    "composition": "%s",
    }}
e.g., metadata: {{
    "composition": "KBe2BO3F2",
    }}
Specifically for composition:
### **Composition Format:**.
    - Due to the phenomenon of homogeneous polycrystals (with the same chemical composition but different crystal structures) in some crystal materials, use prefix to indicate different crystal structures **if there are at least two crystal materials with the same chemical formula/abbreviation explicitly mentioned in the text**.
        - Example: `α-BaHgSnS4`, `β-BaHgSnS4`, `P1-Sr2[B5O8(OH)]2·[B(OH)3]·H2O`, `Pna21-Ba3Mg3(BO3)3F3`
        - Note: sometimes these crystal variants with the same chemical composition but different structures will indicate their crystal forms in parentheses, such as Sr3B14O24 (P21/c). Please record them in the form of P21/c-Sr3B14O24.  
Property section:
[START OF PAPER]\n{text}\n[END OF PAPER]\n

For property specific instruction: 
{instruction}
""")
]
)

nlo_coefficient_instruction = """Extract nonlinear optical property relevant to dij, deff from the text,

Follow these rules:
- For dij and deff, the more specific the data, the higher the priority. For example, d33=4-9 pm V-1 has higher priority than dij=4–9 pm V-1, also d33=10 pm V-1 has higher priority than d33=4–9 pm V-1.
- For dij, if tested under different fundamental light of frequencies (such as @1064nm, @1950nm), only 1064nm needs to be extracted, ignore other frequencies. If there is no provided corresponding test wavelength for dij, just extract dij directly.
- For deff(not dij!!!), if tested under different fundamental light of frequencies (such as @1064nm, @1950nm), extract deff and test wavelength **in a one-to-one correspondence manner, do not confuse the order**. If a deff has no provided corresponding test wavelength, the wavelength extraction result will be 'Null', but don't affect the corresponding sequence relationship between deff and test wavelength!
- Only collect second-order nonlinear susceptibility tensor(dij) and effective nonlinear optical coefficient(deff)
- Prioritize experimental values over calculated values if there is a conflict.
- Prioritize table values over text if there is a conflict. 
- If the value provided is a range, for example, "from 4 pm V-1 to 9 pm V-1", extract it as "4-9 pm V-1".
- If the value is given as "greater than" or "less than", for example, "greater than 4 pm V-1", extract it as ">4 pm V-1".
- If the value is given as "approximately" or "around", for example, "approximately 9 pm v-1", extract it as "≈9 pm V-1".
- Otherwise, extract the value as it is.
"""


class NLOCoefficientExtractor(BaseExtractor):
    target_properties = ['nlo_coefficient']
    context_properties = []
    model = ChatOpenAI(model_name='gpt-4.1', temperature=0)

    def create_model_prompt(self, paragraphs):
        for paragraph in paragraphs:
            paragraph.set_pydantic_model(Records)
            paragraph.set_prompt(simple_prompt_template_no_syn, prompt_vars_dict={
                'instruction': nlo_coefficient_instruction})
            
def load_from_labeled_db(docs):
    return [Paragraph.from_labeled_document(doc, id_) for id_, doc in enumerate(docs)]

extractor = Extraction()
nlo_coefficient_extractor = NLOCoefficientExtractor()
extractor.add_extractors(nlo_coefficient_extractor)

from sisyphus.chain import Writer, Filter
from sisyphus.utils.helper_functions import get_create_resultdb,get_plain_articledb
from sisyphus.chain.chain_elements import run_chains_with_extarction_history_multi_threads

db = get_plain_articledb('test_nlo_0001_labeled')
loader = Filter(db)
result_db = get_create_resultdb('test_nlo_0001_nlo_extracted')
writer = Writer(result_db)

chain = loader + load_from_labeled_db + extractor + writer

run_chains_with_extarction_history_multi_threads(chain=chain, directory='E:\\sisyphus_.0.2.0\\articles_processed', batch_size=10, namespace='test_nlo_0001_nlo_extracted')


e:\sisyphus_.0.2.0\.venv\lib\site-packages\pydantic\main.py:1552: RuntimeWarning: fields may not start with an underscore, ignoring "__tablename__"
  warnings.warn(f'fields may not start with an underscore, ignoring "{f_name}"', RuntimeWarning)


ValueError: no file needed to be extracted